# AdaFace Label-Noise Study: 100k Dataset Generation
This notebook extracts the 100k dataset from raw `.rec` files, applies noise/blur, and saves the output to Google Drive.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Search for train.rec and train.idx
import os
import shutil
from pathlib import Path

print("Searching for train.rec and train.idx in Google Drive...")
train_rec_path = None
train_idx_path = None

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'train.rec' in files:
        train_rec_path = os.path.join(root, 'train.rec')
    if 'train.idx' in files:
        train_idx_path = os.path.join(root, 'train.idx')
        
    if train_rec_path and train_idx_path:
        break

print(f"train.rec found at: {train_rec_path}")
print(f"train.idx found at: {train_idx_path}")

if not train_rec_path or not train_idx_path:
    raise FileNotFoundError("Could not find train.rec or train.idx in Google Drive.")

In [ ]:
%%bash
# 3. Clone Repository and checkout commit
cd /content
rm -rf adaFace-noise-study
git clone https://github.com/Raja2027/adaFace-noise-study.git
cd adaFace-noise-study
git checkout bcb9ae487269decfc377140dd32e75f1172cd847
git submodule update --init
git rev-parse HEAD

In [ ]:
# 4. Copy raw data to repository
print("Copying raw files to repository...")
repo_raw_dir = Path("/content/adaFace-noise-study/data/raw/faces_webface_112x112")
repo_raw_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(train_rec_path, repo_raw_dir / "train.rec")
shutil.copy2(train_idx_path, repo_raw_dir / "train.idx")
print("Raw data copied successfully.")

In [ ]:
# 5. Generate record_label_map.csv if needed
import sys
import pandas as pd
import numpy as np

repo_path = "/content/adaFace-noise-study"
sys.path.append(repo_path)

from scripts.create_100k_dataset import read_idx_file, read_record_header

metadata_dir = Path(repo_path) / "data" / "metadata"
metadata_dir.mkdir(parents=True, exist_ok=True)
label_map_path = metadata_dir / "record_label_map.csv"

if not label_map_path.exists():
    print("record_label_map.csv not found. Generating from raw records...")
    idx = read_idx_file(repo_raw_dir / "train.idx")
    
    records = []
    with open(repo_raw_dir / "train.rec", "rb") as rec_file:
        for record_index, offset in idx.items():
            try:
                res = read_record_header(rec_file, offset)
                if len(res) == 5:
                    flag, label_val, record_id, data_offset, data_length = res
                    if isinstance(label_val, np.ndarray):
                        label_val = label_val[0]
                    records.append({'record_index': record_index, 'y_true': int(label_val)})
            except Exception as e:
                pass
                
    df_label_map = pd.DataFrame(records)
    df_label_map.to_csv(label_map_path, index=False)
    print(f"Generated record_label_map.csv with {len(df_label_map)} records.")
else:
    print("record_label_map.csv already exists.")

In [ ]:
%%bash
# 6. Run Dataset Generation
cd /content/adaFace-noise-study
python scripts/create_100k_dataset.py

In [ ]:
%%bash
# 7. Validate Dataset
cd /content/adaFace-noise-study
python scripts/validate_100k_dataset.py

In [ ]:
%%bash
# 8. Sanity Check
cd /content/adaFace-noise-study
python scripts/sanity_check_100k.py || echo "Sanity check finished/skipped"

In [ ]:
# 9. Copy back to Google Drive
drive_dest = Path("/content/drive/MyDrive/adaFace-noise-study/data")
drive_dest.mkdir(parents=True, exist_ok=True)

print(f"Copying generated dataset to {drive_dest} ... This will take a while.")
!cp -r /content/adaFace-noise-study/data/* /content/drive/MyDrive/adaFace-noise-study/data/
print("Copy completed.")

In [ ]:
# 10. Verify Drive Copy
import json

print("Verifying Drive copy...")
train_dir = drive_dest / "splits" / "100k" / "images" / "train"
heldout_dir = drive_dest / "splits" / "100k" / "images" / "heldout"
lq_train_dir = drive_dest / "corrupted" / "100k_blur_15x15_s5" / "train"

def count_files(directory):
    if not directory.exists(): return 0
    return sum(1 for _ in directory.glob("*") if _.is_file())

num_train = count_files(train_dir)
num_heldout = count_files(heldout_dir)
num_lq = count_files(lq_train_dir)

print(f"HQ Train images: {num_train} (Expected: 100000)")
print(f"HQ Heldout images: {num_heldout} (Expected: 8000)")
print(f"LQ Train images: {num_lq} (Expected: 100000)")

master_csv = drive_dest / "splits" / "100k" / "master_100k.csv"
val_report = drive_dest / "splits" / "100k" / "validation_report.json"

print(f"master_100k.csv exists: {master_csv.exists()}")
print(f"validation_report.json exists: {val_report.exists()}")

if val_report.exists():
    with open(val_report, "r") as f:
        report = json.load(f)
        print("Validation report status:", report.get("status", "UNKNOWN"))
        print("Noise Counts:", report.get("statistics", {}).get("noise_counts", {}))

print("Verification complete.")